# 1.4 — Feature Engineering
**Continued from:** `1.1-jp-eda.ipynb`  
**Purpose:** Explore feature transformations (binning, encoding, scaling, oversampling), select a final feature set, and build a Vertex AI training pipeline using KFP.

---
**Notebook Sections**
1. [Setup](#1-setup)
2. [Experiment 1 — Age Binning Comparison](#2-experiment-1)
3. [Experiment 2 — Scaling](#3-experiment-2)
4. [Experiment 3 — Final Feature Set + Multi-Model Evaluation](#4-experiment-3)
5. [GCP / Vertex AI Integration](#5-gcp)
6. [KFP Pipeline Components](#6-components)
7. [Pipeline Assembly & Deployment](#7-pipeline)


---
## 1. Setup

In [42]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
from google.cloud import aiplatform
from google.cloud import storage
from sklearn.preprocessing import OneHotEncoder

In [43]:
df = pd.read_csv('../data/interim/1.2-edited-data.csv', keep_default_na=False)

df_4 = df.copy().drop(columns=["weight_kg"])

In [44]:
# ---------------------------------------------------------------------------
# Preprocessing helpers
# Used by the exploratory experiment cells below.
# NOTE: KFP components are self-contained closures and duplicate this logic
#       intentionally — they cannot reference notebook-scope functions.
# ---------------------------------------------------------------------------

def clean_prescribed_features(df: pd.DataFrame) -> pd.DataFrame:
    """Coerce medications_prescribed to float and derive is_prescribed flag.
    Also coerces number_of_prior_visits to float.
    Operates on a copy and returns it."""
    df = df.copy()
    df["medications_prescribed"] = df["medications_prescribed"].replace("", pd.NA).astype(float)
    df["is_prescribed"] = df["medications_prescribed"].apply(lambda x: 1 if x > 0 else 0)
    df["number_of_prior_visits"] = df["number_of_prior_visits"].replace("", pd.NA).astype(float)
    return df


def add_length_of_stay_score(df: pd.DataFrame) -> pd.DataFrame:
    """Converts length_of_stay (days) to an ordinal risk score (1–7)."""
    df = df.copy()
    def _score(x):
        if x <= 1: return 1
        if x <= 2: return 2
        if x <= 3: return 3
        if x <= 6: return 4
        if x <= 14: return 5
        return 7
    df["length_of_stay_score"] = df["length_of_stay"].apply(_score)
    return df


### Goals
- Evaluate binning strategies for `age` and `length_of_stay`
- Determine how to handle sparse/missing values in `medications_prescribed` and `number_of_prior_visits`
- Compare model performance across feature engineering strategies
- Finalise a feature set for the production training pipeline

---
## 2. Experiment 1 — Age Binning Comparison

Compare raw `age` vs. age bins as a feature using a Random Forest baseline (no scaling, no oversampling).

In [45]:
## binning age

age_bins_1 = [0, 18, 25, 40, 65, 80, 100]

df_4["age_bin_1"] = pd.cut(df_4["age"], bins=age_bins_1)

df_4["age_bin_1"].value_counts()

age_bin_1
(40, 65]     4008
(25, 40]     1693
(65, 80]     1197
(18, 25]      376
(80, 100]     350
(0, 18]       324
Name: count, dtype: int64

In [46]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

model = RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        random_state=42,
    )

In [47]:
df_bins = df_4.copy().drop(columns=["age"])
df_nonbins = df_4.copy().drop(columns=["age_bin_1"])

df_bins = pd.get_dummies(df_bins, drop_first=True)
df_nonbins = pd.get_dummies(df_nonbins, drop_first=True)

In [48]:
X_bins_train, X_bins_test, y_bins_train, y_bins_test = train_test_split(
    df_bins.drop(columns=["target"]),
    df_bins["target"],
    test_size=0.2,
    random_state=42,
)

model.fit(X_bins_train, y_bins_train)

preds = model.predict(X_bins_test)

print(classification_report(y_bins_test, preds))

print(confusion_matrix(y_bins_test, preds))

              precision    recall  f1-score   support

           0       0.86      1.00      0.92      1316
           1       0.94      0.24      0.38       274

    accuracy                           0.87      1590
   macro avg       0.90      0.62      0.65      1590
weighted avg       0.88      0.87      0.83      1590

[[1312    4]
 [ 209   65]]


In [49]:
X_nonbins_train, X_nonbins_test, y_nonbins_train, y_nonbins_test = train_test_split(
    df_nonbins.drop(columns=["target"]),
    df_nonbins["target"],
    test_size=0.2,
    random_state=42,
)
model.fit(X_nonbins_train, y_nonbins_train)

preds_nonbins = model.predict(X_nonbins_test)

print(classification_report(y_nonbins_test, preds_nonbins))

print(confusion_matrix(y_nonbins_test, preds_nonbins))

              precision    recall  f1-score   support

           0       0.86      1.00      0.92      1316
           1       1.00      0.19      0.32       274

    accuracy                           0.86      1590
   macro avg       0.93      0.60      0.62      1590
weighted avg       0.88      0.86      0.82      1590

[[1316    0]
 [ 221   53]]


### Observations — Experiment 1

- Age binning performs slightly better than raw age on an unscaled, unbalanced dataset.
- Both configurations still reflect the class imbalance problem — the next step is to normalise features and then re-evaluate.

---
## 3. Experiment 2 — MinMax Scaling

Apply `MinMaxScaler` to the binned and non-binned splits to see if normalisation improves performance.

In [50]:
from sklearn.preprocessing import MinMaxScaler

cols_to_scale_binned = ["height_m", "bmi", "adjusted_weight_kg", "length_of_stay"]
cols_to_scale_non_binned = ["height_m", "bmi", "adjusted_weight_kg", "length_of_stay", "age"]

bins_scaler = MinMaxScaler()
non_bins_scaler = MinMaxScaler()

X_bins_train[cols_to_scale_binned] = bins_scaler.fit_transform(X_bins_train[cols_to_scale_binned])
X_nonbins_train[cols_to_scale_non_binned] = non_bins_scaler.fit_transform(X_nonbins_train[cols_to_scale_non_binned])

In [51]:
model.fit(X_bins_train, y_bins_train)
preds_scaled_bins = model.predict(X_bins_test)
print(classification_report(y_bins_test, preds_scaled_bins))


model.fit(X_nonbins_train, y_nonbins_train)
preds_scaled_nonbins = model.predict(X_nonbins_test)
print(classification_report(y_nonbins_test, preds_scaled_nonbins))

              precision    recall  f1-score   support

           0       0.83      1.00      0.91      1316
           1       1.00      0.01      0.01       274

    accuracy                           0.83      1590
   macro avg       0.91      0.50      0.46      1590
weighted avg       0.86      0.83      0.75      1590

              precision    recall  f1-score   support

           0       0.83      1.00      0.91      1316
           1       1.00      0.01      0.02       274

    accuracy                           0.83      1590
   macro avg       0.91      0.51      0.46      1590
weighted avg       0.86      0.83      0.75      1590



## Experiment 3 — Revised Feature Set

Rebuilding from `df_4` with a broader set of transformations:
- Labelled age bins instead of numeric bins
- Ordinal `length_of_stay_score` instead of categorical bins
- SMOTE oversampling
- Multiple classifiers for comparison

In [52]:
df = df_4.copy()

df = df.drop(columns = ["age"])

In [53]:
length_of_stay_categories = {
    "0-3": (0, 3),
    "4-7": (4, 7),
    "8-14": (8, 14),
    "15-30": (15, 30),
    "31+": (31, np.inf)
}

df["length_of_stay_cat"] = pd.cut(df["length_of_stay"], bins=[0, 3, 7, 14, 30, np.inf], labels=length_of_stay_categories.keys(), right=False)   

In [54]:
df = clean_prescribed_features(df)


In [55]:
df_features = df[[
    "gender",
    "age_bin_1",
    "height_m",
    "bmi",
    "adjusted_weight_kg",
    "length_of_stay_cat",
    "is_prescribed",
    "number_of_prior_visits",
    "smoker",
    "has_diabetes",
    "has_hypertension",
    "exercise_frequency",
    "diet_type",
    "type_of_treatment",
    "target"
]]

df_features = df_features.dropna(inplace=False)

In [56]:
df_feats = df_features.copy().drop(columns=["target"])
target = df_features["target"]
df_features = pd.get_dummies(df_features, drop_first=True)

df_features

,height_m,bmi,adjusted_weight_kg,is_prescribed,number_of_prior_visits,smoker,has_diabetes,has_hypertension,target,gender_Male,...,length_of_stay_cat_15-30,length_of_stay_cat_31+,exercise_frequency_Occasional,exercise_frequency_Regular,diet_type_High-fat,diet_type_Other,diet_type_Vegetarian,type_of_treatment_Minor Surgery,type_of_treatment_None,type_of_treatment_Other Treatment
0,1.6,25.0,63.283346,1,3.0,False,False,False,0,False,...,False,False,False,True,True,False,False,False,True,False
1,1.8,27.0,87.678859,0,2.0,True,False,False,0,False,...,False,False,False,True,True,False,False,False,True,False
3,1.8,27.7,89.717694,0,3.0,False,False,False,0,False,...,False,False,False,False,False,True,False,False,False,False
4,1.9,22.4,80.528927,1,7.0,False,False,False,1,False,...,False,False,True,False,True,False,False,False,False,False
5,1.8,29.1,94.467851,0,3.0,False,False,False,0,False,...,False,False,False,True,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7943,1.6,19.1,49.881959,1,4.0,False,False,False,0,True,...,False,False,False,True,False,False,True,False,False,False
7944,1.7,20.1,57.904835,1,0.0,False,True,False,0,False,...,False,False,True,False,False,False,False,True,False,False
7945,1.7,28.5,82.058378,1,1.0,True,False,False,0,True,...,False,False,True,False,True,False,False,True,False,False
7946,1.7,25.7,74.046629,1,5.0,True,False,False,0,False,...,False,False,False,True,False,False,False,True,False,False


In [57]:
from sklearn.preprocessing import StandardScaler

cols_to_scale = ["height_m", "bmi", "adjusted_weight_kg", "number_of_prior_visits"]

df_feats = pd.get_dummies(df_feats, drop_first=True)

df_feats_smote = df_feats.copy()


X_train, X_test, y_train, y_test = train_test_split(
    df_feats,
    target,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()

X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])

model.fit(X_train, y_train)

preds = model.predict(X_test)
print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           0       0.84      1.00      0.91      1272
           1       0.83      0.02      0.04       256

    accuracy                           0.84      1528
   macro avg       0.83      0.51      0.47      1528
weighted avg       0.83      0.84      0.76      1528



In [58]:
from imblearn.over_sampling import SMOTE, RandomOverSampler

ros = RandomOverSampler(random_state=42)

X_train_ros, X_test_ros, y_train_ros, y_test_ros = train_test_split(df_feats_smote, target, test_size=0.2, random_state=42, stratify=target)

X_train_ros, y_train_ros = ros.fit_resample(X_train_ros, y_train_ros)

X_train_ros[cols_to_scale] = scaler.fit_transform(X_train_ros[cols_to_scale])
model.fit(X_train_ros, y_train_ros)

preds_ros = model.predict(X_test_ros)
print(classification_report(y_test_ros, preds_ros))

print(confusion_matrix(y_test_ros, preds_ros))


              precision    recall  f1-score   support

           0       0.83      1.00      0.91      1263
           1       0.00      0.00      0.00       265

    accuracy                           0.83      1528
   macro avg       0.41      0.50      0.45      1528
weighted avg       0.68      0.83      0.75      1528

[[1263    0]
 [ 265    0]]


/opt/anaconda3/envs/vertex-readmissions-env/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/vertex-readmissions-env/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/vertex-readmissions-env/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [59]:
from sklearn.ensemble import AdaBoostClassifier

ada_model = AdaBoostClassifier(
    n_estimators=250,
    learning_rate=0.01,
    random_state=42,
)   

ada_model.fit(X_train_ros, y_train_ros)
preds_ada = ada_model.predict(X_test_ros)
print(classification_report(y_test_ros, preds_ada))
    

              precision    recall  f1-score   support

           0       0.89      0.95      0.92      1263
           1       0.64      0.46      0.54       265

    accuracy                           0.86      1528
   macro avg       0.77      0.70      0.73      1528
weighted avg       0.85      0.86      0.85      1528



### Observations — Experiment 2

Random Forest is not predicting the minority class even with balanced data. The scaler was fit on the original training set and not re-applied to the ROS-resampled features — investigating in Experiment 3.

In [60]:
X_train_ros

,height_m,bmi,adjusted_weight_kg,is_prescribed,number_of_prior_visits,smoker,has_diabetes,has_hypertension,gender_Male,"age_bin_1_(18, 25]",...,length_of_stay_cat_15-30,length_of_stay_cat_31+,exercise_frequency_Occasional,exercise_frequency_Regular,diet_type_High-fat,diet_type_Other,diet_type_Vegetarian,type_of_treatment_Minor Surgery,type_of_treatment_None,type_of_treatment_Other Treatment
0,-0.957186,0.277765,-0.389017,1,1.418077,False,True,False,True,False,...,False,False,True,False,True,False,False,False,False,False
1,-0.003683,0.034965,0.049374,1,0.341526,False,False,False,False,False,...,False,False,False,True,True,False,False,False,False,False
2,0.949821,1.999436,2.445903,0,-0.735025,False,False,False,True,False,...,False,False,False,False,True,False,False,False,False,False
3,-0.957186,-1.841215,-1.872720,1,-0.735025,False,False,True,False,False,...,False,False,True,False,False,False,False,True,False,False
4,-0.957186,0.962019,0.189215,1,-0.735025,False,False,False,False,False,...,False,False,True,False,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10093,-0.957186,-0.207835,-0.694712,1,-0.196749,False,False,False,False,False,...,False,False,True,False,True,False,False,False,False,True
10094,-0.957186,0.233619,-0.370676,1,0.341526,True,False,True,False,False,...,False,False,False,False,False,True,False,False,True,False
10095,-0.957186,-0.229908,-0.712109,1,0.341526,True,False,False,False,False,...,False,False,True,False,False,True,False,False,False,False
10096,-0.957186,0.299837,-0.344032,1,-0.196749,False,True,True,False,False,...,False,False,False,False,True,False,False,False,False,True


### Models to Evaluate

1. Logistic Regression — baseline
2. Random Forest — tune hyperparameters
3. AdaBoost
4. Gradient Boosting (XGBoost-style via sklearn)

### Experiment 3 — Revised Encodings

Switching `length_of_stay` to an ordinal score and using labelled age bins to improve interpretability.

In [61]:
df = df_4.copy()

age_bins_label = {
    "0-18": (0, 18),
    "19-25": (19, 25),
    "26-40": (26, 40),
    "41-65": (41, 65),
    "66-80": (66, 80),
    "81+": (81, np.inf)
}

df["age_bin_label"] = pd.cut(df["age"], bins=[0, 18, 25, 40, 65, 80, np.inf], labels=age_bins_label.keys(), right=False)

In [62]:
df.drop(columns=["age_bin_1"], inplace=True)

In [63]:
df = clean_prescribed_features(df)


In [64]:
df = add_length_of_stay_score(df)


In [65]:
df = df.dropna(inplace=False)
target = df["target"]
df_features = df.drop(columns=["age", "length_of_stay", "medications_prescribed", "patient_id", "target"])


In [66]:
df_features = pd.get_dummies(df_features, drop_first=True)

cols_to_scale = ["height_m", "bmi", "adjusted_weight_kg", "number_of_prior_visits", "length_of_stay_score"]


X_train, X_test, y_train, y_test = train_test_split(
    df_features,
    target,
    test_size=0.2,
    random_state=42,
    stratify=target
)


smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

scaler = StandardScaler()

X_train_smote[cols_to_scale] = scaler.fit_transform(X_train_smote[cols_to_scale])
X_train[cols_to_scale] = scaler.transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])


In [67]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier

rf_model = RandomForestClassifier(
    n_estimators=250,
    max_depth=8,
    random_state=42,
)

ada_model = AdaBoostClassifier(
    n_estimators=250,
    learning_rate=0.001,
    random_state=42,
)
log_reg_model = LogisticRegression(
    random_state=42,
    max_iter=1000,
)

xgb_model = GradientBoostingClassifier(
    n_estimators=250,
    learning_rate=0.01,
    max_depth=5,
    random_state=42,
)

rf_model.fit(X_train_smote, y_train_smote)
ada_model.fit(X_train_smote, y_train_smote)
log_reg_model.fit(X_train_smote, y_train_smote)
xgb_model.fit(X_train_smote, y_train_smote)

,"loss loss: {'log_loss', 'exponential'}, default='log_loss'The loss function to be optimized. 'log_loss' refers to binomial andmultinomial deviance, the same as used in logistic regression.It is a good choice for classification with probabilistic outputs.For loss 'exponential', gradient boosting recovers the AdaBoost algorithm.",'log_loss'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.For an example of the effects of this parameter and its interaction with``subsample``, see:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_regularization.py`.",0.01
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",250
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are'friedman_mse' for the mean squared error with improvement score byFriedman, 'squared_error' for mean squared error. The default value of'friedman_mse' is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",5
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``,

In [68]:
rf_preds = rf_model.predict(X_test)
ada_preds = ada_model.predict(X_test)

print("Random Forest Classification Report:")
print(classification_report(y_test, rf_preds))

print("AdaBoost Classification Report:")
print(classification_report(y_test, ada_preds))

log_reg_preds = log_reg_model.predict(X_test)
print("Logistic Regression Classification Report:")
print(classification_report(y_test, log_reg_preds))

xgb_preds = xgb_model.predict(X_test)
print("XGBoost Classification Report:")
print(classification_report(y_test, xgb_preds))

Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.95      0.94      1165
           1       0.73      0.69      0.71       237

    accuracy                           0.90      1402
   macro avg       0.83      0.82      0.82      1402
weighted avg       0.90      0.90      0.90      1402

AdaBoost Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.98      0.92      1165
           1       0.67      0.22      0.34       237

    accuracy                           0.85      1402
   macro avg       0.77      0.60      0.63      1402
weighted avg       0.83      0.85      0.82      1402

Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.94      0.95      1165
           1       0.72      0.75      0.74       237

    accuracy                           0.91      1402
   macro avg    

### Results — Experiment 3

> **Note:** Test set must be transformed with the scaler fit on training data — not re-fit. This is handled correctly in the KFP `apply_preprocessing` component below.

---
## GCP / Vertex AI Integration

Feature engineering is baked into the training pipeline rather than versioned as a dataset transformation.  
The pipeline handles: data validation → train/val split → oversampling → preprocessing (fit + apply) → model training → evaluation.

In [69]:
""" first going to make a v1.4 dataset that just removes weight because thats the only constant and will be
going forward i believe"""

import sys
sys.path.insert(0, "..")
from scripts.gcs_utils import log_dataset_to_gcs, log_pipeline_run
from pathlib import Path

df = pd.read_csv('../data/interim/1.2-edited-data.csv', keep_default_na=False)

df_4 = df.copy().drop(columns=["weight_kg"])

df_4.to_csv('../data/interim/1.4-edited-data.csv', index=False)


In [70]:
PROJECT_ID = "readmission-543-project"
LOCATION = "us-central1"
BUCKET_ROOT_URI = "gs://readmission-bucket"


aiplatform.init(project=PROJECT_ID, location=LOCATION)

log_dataset_to_gcs(
    DATASET_LOCAL_PATH=Path("../data/interim/1.4-edited-data.csv"),
    VERSION_ID="v1.4",
    BUCKET_ROOT_URI=BUCKET_ROOT_URI,
    PROJECT_ID=PROJECT_ID,
    LOCATION=LOCATION,
    log_experiment=True,
    EXPERIMENT_NAME="readmissions-data-versions",
    description="Dataset with initial cleaning+formatting, as well as weight_kg feature removed.",
    tags=["initial-clean", "v1.4", "weight-removed"],
    resume_run=True,
)

Uploaded: gs://readmission-bucket/datasets/readmissions/v1.4/train.csv
Uploaded: gs://readmission-bucket/datasets/readmissions/v1.4/manifest.json


Logged dataset version to Vertex Experiments: readmissions-data-versions / readmissions-data-v1-4


In [71]:
df_4

,patient_id,age,gender,ethnicity,hospital_id,height_m,smoker,bmi,adjusted_weight_kg,has_diabetes,has_hypertension,exercise_frequency,diet_type,number_of_prior_visits,medications_prescribed,length_of_stay,type_of_treatment,target
0,1000000,23,Female,African American,Hosp2,1.6,False,25.0,63.283346,False,False,Regular,High-fat,3.0,3.0,0,None,0
1,1000002,56,Female,Hispanic,Hosp3,1.8,True,27.0,87.678859,False,False,Regular,High-fat,2.0,,2,None,0
2,1000003,28,Male,African American,Hosp1,1.8,False,35.0,113.497844,False,True,None,Other,,2.0,5,None,0
3,1000004,70,Female,Caucasian,Hosp2,1.8,False,27.7,89.717694,False,False,None,Other,3.0,,0,Major Surgery,0
4,1000005,48,Female,Hispanic,Hosp1,1.9,False,22.4,80.528927,False,False,Occasional,High-fat,7.0,5.0,7,Major Surgery,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7943,1009991,30,Male,Caucasian,Hosp2,1.6,False,19.1,49.881959,False,False,Regular,Vegetarian,4.0,1.0,3,Major Surgery,0
7944,1009992,45,Female,Caucasian,Hosp3,1.7,False,20.1,57.904835,True,False,Occasional,Balanced,0.0,2.0,2,Minor Surgery,0
7945,1009994,18,Male,Caucasian,Hosp1,1.7,True,28.5,82.058378,False,False,Occasional,High-fat,1.0,4.0,8,Minor Surgery,0
7946,1009998,37,Female,Caucasian,Hosp2,1.7,True,25.7,74.046629,False,False,Regular,Balanced,5.0,4.0,1,Minor Surgery,0


---
## 6. KFP Pipeline Components

Each component below is a self-contained KFP v2 `@component` decorated function. They are combined into the pipeline in Section 7.

### Component 1 — `load_validate_data`

Reads a CSV from GCS, validates required columns and target distribution, and passes the dataset downstream.

In [72]:
from kfp.v2 import dsl
from kfp.v2.dsl import component, Output, Dataset, Input, Model, Metrics, Artifact
from google.cloud import aiplatform


@component(packages_to_install=["pandas", "numpy", "fsspec", "gcsfs"])
def load_validate_data(input_dataset_path: str, output_dataset: Output[Dataset]):
    import pandas as pd
    import numpy as np

    df = pd.read_csv(input_dataset_path)

    # --- Basic dataset validation ---
    if df.empty:
        raise ValueError("Input dataset is empty")

    print(f"Dataset shape: {df.shape}")

    # Drop accidental CSV index columns if present
    df = df.loc[:, ~df.columns.str.contains(r"^Unnamed")]

    # --- Required pipeline columns ---
    target_col = "target"
    id_col = "patient_id"

    required_cols = [target_col, id_col]
    missing_required = [col for col in required_cols if col not in df.columns]
    if missing_required:
        raise ValueError(f"Missing required columns: {missing_required}")

    # --- Target validation ---
    print("Target distribution:")
    print(df[target_col].value_counts(dropna=False))

    if df[target_col].dropna().nunique() < 2:
        raise ValueError(
            f"Target column '{target_col}' must contain at least two classes"
        )

    # --- Save output ---
    df.to_csv(output_dataset.path, index=False)

/var/folders/9w/27dwnd7s3plbp9xj1v9qrn5m0000gn/T/ipykernel_43722/2649678376.py:6: FutureWarning: The default base_image used by the @dsl.component decorator will switch from 'python:3.11' to 'python:3.12' on Oct 1, 2027. To ensure your existing components work with versions of the KFP SDK released after that date, you should provide an explicit base_image argument and ensure your component works as intended on Python 3.12.
  @component(packages_to_install=["pandas", "numpy", "fsspec", "gcsfs"])


### Component 2 — `split_data`

Stratified train/validation split. Preserves `patient_id` throughout.

In [73]:
@component(packages_to_install=["pandas", "scikit-learn"])
def split_data(
    input_dataset: Input[Dataset],
    train_dataset: Output[Dataset],
    validation_dataset: Output[Dataset],
    test_size: float = 0.2,
    random_state: int = 42,
):

    from sklearn.model_selection import train_test_split
    import pandas as pd

    df = pd.read_csv(input_dataset.path)

    target_col = "target"
    id_col = "patient_id"

    X = df.drop(columns=[target_col, id_col])
    y = df[target_col]
    ids = df[id_col]

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=random_state
    )

    train_df = pd.concat([X_train, y_train, ids.loc[X_train.index]], axis=1)
    val_df = pd.concat([X_val, y_val, ids.loc[X_val.index]], axis=1)

    train_df.to_csv(train_dataset.path, index=False)
    val_df.to_csv(validation_dataset.path, index=False)

    print(f"Training set shape: {train_df.shape}")
    print(f"Validation set shape: {val_df.shape}")

    print("Training target distribution:")
    print(train_df[target_col].value_counts(normalize=True))

    print("Validation target distribution:")
    print(val_df[target_col].value_counts(normalize=True))

/var/folders/9w/27dwnd7s3plbp9xj1v9qrn5m0000gn/T/ipykernel_43722/831733875.py:1: FutureWarning: The default base_image used by the @dsl.component decorator will switch from 'python:3.11' to 'python:3.12' on Oct 1, 2027. To ensure your existing components work with versions of the KFP SDK released after that date, you should provide an explicit base_image argument and ensure your component works as intended on Python 3.12.
  @component(packages_to_install=["pandas", "scikit-learn"])


### Component 3 — `oversample_training`

Applies `RandomOverSampler` to the training split only to address class imbalance.

In [74]:
@component(
    packages_to_install=[
        "pandas",
        "numpy",
        "scikit-learn",
        "imbalanced-learn"
    ]
)
def oversample_training(
    input_dataset: Input[Dataset],
    output_dataset: Output[Dataset],
    target_col: str = "target",
    id_col: str = "patient_id",
    random_state: int = 42
):
    import pandas as pd
    from imblearn.over_sampling import RandomOverSampler

    # --- Load dataset ---
    df = pd.read_csv(input_dataset.path)

    print(f"Input training dataset shape: {df.shape}")

    # --- Validate required columns ---
    if target_col not in df.columns:
        raise ValueError(f"Missing target column: {target_col}")

    if id_col not in df.columns:
        raise ValueError(f"Missing ID column: {id_col}")

    # --- Check class distribution before ---
    print("Class distribution BEFORE oversampling:")
    print(df[target_col].value_counts(dropna=False))

    # --- Separate components ---
    # Note: ID is kept as part of the dataset but not used for modeling logic
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    # --- Apply Random Oversampling ---
    ros = RandomOverSampler(random_state=random_state)

    X_resampled, y_resampled = ros.fit_resample(X, y)

    # --- Reconstruct dataset ---
    resampled_df = X_resampled.copy()
    resampled_df[target_col] = y_resampled

    # --- Log results ---
    print("Class distribution AFTER oversampling:")
    print(resampled_df[target_col].value_counts(dropna=False))

    print(f"Original dataset shape: {df.shape}")
    print(f"Oversampled dataset shape: {resampled_df.shape}")

    # --- Save output ---
    resampled_df.to_csv(output_dataset.path, index=False)

/var/folders/9w/27dwnd7s3plbp9xj1v9qrn5m0000gn/T/ipykernel_43722/174125770.py:1: FutureWarning: The default base_image used by the @dsl.component decorator will switch from 'python:3.11' to 'python:3.12' on Oct 1, 2027. To ensure your existing components work with versions of the KFP SDK released after that date, you should provide an explicit base_image argument and ensure your component works as intended on Python 3.12.
  @component(


### Component 4 — `fit_apply_preprocessing`

Fits all preprocessing steps (imputation, scaling, one-hot encoding) on the training split and saves artifacts for reuse by the validation component.

In [75]:
@component(packages_to_install=["pandas", "numpy", "scikit-learn", "joblib"])
def fit_apply_preprocessing(
    input_dataset: Input[Dataset],
    output_dataset: Output[Dataset],
    preprocessing_artifacts: Output[Artifact],
    target_col: str = "target",
    id_col: str = "patient_id",
):
    import os
    import json
    import joblib
    import numpy as np
    import pandas as pd
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler

    # -----------------------------
    # Load training dataset
    # -----------------------------
    df = pd.read_csv(input_dataset.path)

    print(f"Input training dataset shape: {df.shape}")

    # -----------------------------
    # Separate ID and target
    # Keep them outside the fitted preprocessing logic
    # -----------------------------
    ids = df[[id_col]].copy()
    y = df[[target_col]].copy()
    X = df.drop(columns=[id_col, target_col]).copy()

    # -----------------------------
    # Fixed-rule preprocessing
    # -----------------------------

    X["age_group"] = pd.cut(
        X["age"],
        bins=[0, 18, 25, 40, 65, 80, np.inf],
        labels=["0-18", "19-25", "26-40", "41-65", "66-80", "81+"],
        right=False,
    )
    X = X.drop(columns=["age"])

    X["medications_prescribed"] = X["medications_prescribed"].replace("", pd.NA)
    X["medications_prescribed"] = X["medications_prescribed"].astype(float)

    X["medications_prescribed"] = X["medications_prescribed"].apply(
        lambda x: 1 if x > 0 else 0
    )

    X["number_of_prior_visits"] = X["number_of_prior_visits"].replace("", pd.NA)
    X["number_of_prior_visits"] = X["number_of_prior_visits"].astype(float)

    X["length_of_stay_score"] = X["length_of_stay"].apply(
        lambda x: (
            1
            if x <= 1
            else (
                2
                if x <= 2
                else (3 if x <= 3 else (4 if x <= 6 else (5 if x <= 14 else 7)))
            )
        )
    )
    X.drop(columns=["length_of_stay"], inplace=True)
    
    num_cols = ["height_m", "bmi", "adjusted_weight_kg", "number_of_prior_visits", "length_of_stay_score"]
    cat_cols = X.select_dtypes(include=["object", "category", "str"]).columns.tolist()
    # boolean handling taken out becasue fml add it back later ^_^

    # -----------------------------
    # Learned preprocessing
    # -----------------------------
    
    #impute median on numerics
    
    median_imputer = SimpleImputer(strategy="median", missing_values= pd.NA)
    mode_imputer = SimpleImputer(strategy="most_frequent", missing_values= pd.NA)
    
    
    X[num_cols] = median_imputer.fit_transform(X[num_cols])
    X[cat_cols] = mode_imputer.fit_transform(X[cat_cols])
    
    #standardize numeric cols
    scaler = StandardScaler()
    X[num_cols] = scaler.fit_transform(X[num_cols])
    
    # one hot encode categories
    X = pd.get_dummies(X, drop_first=True)
    
    # Reconstruct final training dataset
    # -----------------------------
    prepared_df = pd.concat([ids, X, y], axis=1)

    print(f"Prepared training dataset shape: {prepared_df.shape}")

    # -----------------------------
    # Save transformed dataset
    # -----------------------------
    prepared_df.to_csv(output_dataset.path, index=False)

    # -----------------------------
    # Save preprocessing artifacts
    # -----------------------------
    os.makedirs(preprocessing_artifacts.path, exist_ok=True)

    # Save encoder
    joblib.dump(
        scaler, os.path.join(preprocessing_artifacts.path, "standard_scaler.joblib")
    )
    
    # Save imputers
    
    joblib.dump(
        median_imputer, os.path.join(preprocessing_artifacts.path, "median_imputer.joblib")
    )
    joblib.dump(
        mode_imputer, os.path.join(preprocessing_artifacts.path, "mode_imputer.joblib")
    )
    
    # Save preprocessing metadata
    metadata = {
        "target_col": 'target',
        "id_col": 'patient_id',
        "num_cols": num_cols,
        "categorical_cols": cat_cols,
        "output_feature_columns": X.columns.tolist(),

    }
    with open(
        os.path.join(preprocessing_artifacts.path, "preprocessing_metadata.json"), "w"
    ) as f:
        json.dump(metadata, f, indent=2)

    print("Preprocessing artifacts saved:")
    print(os.listdir(preprocessing_artifacts.path))

/var/folders/9w/27dwnd7s3plbp9xj1v9qrn5m0000gn/T/ipykernel_43722/1728074349.py:1: FutureWarning: The default base_image used by the @dsl.component decorator will switch from 'python:3.11' to 'python:3.12' on Oct 1, 2027. To ensure your existing components work with versions of the KFP SDK released after that date, you should provide an explicit base_image argument and ensure your component works as intended on Python 3.12.
  @component(packages_to_install=["pandas", "numpy", "scikit-learn", "joblib"])


### Component 5 — `apply_preprocessing`

Applies the saved preprocessing artifacts from Component 4 to the validation split. Realigns feature columns to match the training schema.

In [76]:
@component(
    packages_to_install=[
        "pandas",
        "numpy",
        "scikit-learn",
        "joblib"
    ]
)
def apply_preprocessing(
    input_dataset: Input[Dataset],
    preprocessing_artifacts: Input[Artifact],
    output_dataset: Output[Dataset],
    target_col: str = "target",
    id_col: str = "patient_id"
):
    import os
    import json
    import joblib
    import numpy as np
    import pandas as pd

    # -----------------------------
    # Load validation dataset
    # -----------------------------
    df = pd.read_csv(input_dataset.path)

    print(f"Input validation dataset shape: {df.shape}")

    # -----------------------------
    # Load preprocessing artifacts
    # -----------------------------
    metadata_path = os.path.join(preprocessing_artifacts.path, "preprocessing_metadata.json")
    median_imputer_path = os.path.join(preprocessing_artifacts.path, "median_imputer.joblib")
    mode_imputer_path = os.path.join(preprocessing_artifacts.path, "mode_imputer.joblib")
    scaler_path = os.path.join(preprocessing_artifacts.path, "standard_scaler.joblib")

    if not os.path.exists(metadata_path):
        raise ValueError(f"Missing preprocessing metadata file: {metadata_path}")

    if not os.path.exists(scaler_path):
        raise ValueError(f"Missing scaler artifact file: {scaler_path}")

    with open(metadata_path, "r") as f:
        metadata = json.load(f)

    median_imputer = joblib.load(median_imputer_path)
    mode_imputer = joblib.load(mode_imputer_path)
    scaler = joblib.load(scaler_path)
    
    
    num_cols = metadata["num_cols"]
    categorical_cols = metadata["categorical_cols"]
    expected_output_cols = metadata["output_feature_columns"]

    print("Numeric columns to impute and scale:")
    print(num_cols)
    print("Categorical columns to encode:")
    print(categorical_cols)

    # -----------------------------
    # Separate ID and target
    # -----------------------------
    ids = df[[id_col]].copy()
    y = df[[target_col]].copy()
    X = df.drop(columns=[id_col, target_col]).copy()

    # -----------------------------
    # Apply same fixed-rule preprocessing
    # -----------------------------
    
    X["age_group"] = pd.cut(
        X["age"],
        bins=[0, 18, 25, 40, 65, 80, np.inf],
        labels=["0-18", "19-25", "26-40", "41-65", "66-80", "81+"],
        right=False,
    )
    X = X.drop(columns=["age"])

    X["medications_prescribed"] = X["medications_prescribed"].replace("", pd.NA)
    X["medications_prescribed"] = X["medications_prescribed"].astype(float)

    X["medications_prescribed"] = X["medications_prescribed"].apply(
        lambda x: 1 if x > 0 else 0
    )

    X["number_of_prior_visits"] = X["number_of_prior_visits"].replace("", pd.NA)
    X["number_of_prior_visits"] = X["number_of_prior_visits"].astype(float)

    X["length_of_stay_score"] = X["length_of_stay"].apply(
        lambda x: (
            1
            if x <= 1
            else (
                2
                if x <= 2
                else (3 if x <= 3 else (4 if x <= 6 else (5 if x <= 14 else 7)))
            )
        )
    )
    X.drop(columns=["length_of_stay"], inplace=True)

    # -----------------------------
    # Apply learned preprocessing
    # -----------------------------

    # Impute and scale numerics
    X[num_cols] = median_imputer.transform(X[num_cols])
    X[num_cols] = scaler.transform(X[num_cols])
    
    # impute categories
    X[categorical_cols] = mode_imputer.transform(X[categorical_cols])

    #encode
    X = pd.get_dummies(X, drop_first=True) #this may need to be changed in the future to ensure consistency but idk
    

    # -----------------------------
    # Align validation columns to training columns
    # -----------------------------
    # Important safeguard: ensure exact feature-space match with training output
    X_prepared = X.reindex(columns=expected_output_cols, fill_value=0)

    # -----------------------------
    # Reconstruct final validation dataset
    # -----------------------------
    prepared_df = pd.concat([ids, X_prepared, y], axis=1)

    print(f"Prepared validation dataset shape: {prepared_df.shape}")

    # -----------------------------
    # Save output dataset
    # -----------------------------
    prepared_df.to_csv(output_dataset.path, index=False)

/var/folders/9w/27dwnd7s3plbp9xj1v9qrn5m0000gn/T/ipykernel_43722/1858332830.py:1: FutureWarning: The default base_image used by the @dsl.component decorator will switch from 'python:3.11' to 'python:3.12' on Oct 1, 2027. To ensure your existing components work with versions of the KFP SDK released after that date, you should provide an explicit base_image argument and ensure your component works as intended on Python 3.12.
  @component(


### Component 6 — `train_model`

Trains a `RandomForestClassifier` on the preprocessed training data and saves the model artifact + metadata.

In [77]:
@component(
    packages_to_install=[
        "pandas",
        "numpy",
        "scikit-learn",
        "joblib"
    ]
)
def train_model(
    input_dataset: Input[Dataset],
    model_artifact: Output[Artifact],
    target_col: str = "target",
    id_col: str = "patient_id",
    n_estimators: int = 250,
    max_depth: int = 8,
    random_state: int = 42
):
    import os
    import json
    import joblib
    import pandas as pd
    from sklearn.ensemble import RandomForestClassifier

    # -----------------------------
    # Load prepared training dataset
    # -----------------------------
    df = pd.read_csv(input_dataset.path)

    print(f"Training dataset shape: {df.shape}")


    # -----------------------------
    # Separate features and target
    # -----------------------------
    feature_cols = [col for col in df.columns if col not in [id_col, target_col]]

    if len(feature_cols) == 0:
        raise ValueError("No feature columns available for model training")

    X_train = df[feature_cols]
    y_train = df[target_col]

    print(f"Number of training rows: {len(df)}")
    print(f"Number of features: {len(feature_cols)}")

    print("Training target distribution:")
    print(y_train.value_counts(dropna=False))

    # -----------------------------
    # Fit model
    # -----------------------------
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=random_state,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    print("Model training complete.")

    # -----------------------------
    # Save model artifact
    # -----------------------------
    os.makedirs(model_artifact.path, exist_ok=True)

    joblib.dump(model, os.path.join(model_artifact.path, "random_forest_model.joblib"))

    metadata = {
        "model_type": "RandomForestClassifier",
        "target_col": target_col,
        "id_col": id_col,
        "feature_cols": feature_cols,
        "n_estimators": n_estimators,
        "max_depth": max_depth,
        "random_state": random_state,
        "n_training_rows": int(len(df)),
        "n_features": int(len(feature_cols))
    }

    with open(os.path.join(model_artifact.path, "model_metadata.json"), "w") as f:
        json.dump(metadata, f, indent=2)

    print("Saved model artifact contents:")
    print(os.listdir(model_artifact.path))

/var/folders/9w/27dwnd7s3plbp9xj1v9qrn5m0000gn/T/ipykernel_43722/2778173525.py:1: FutureWarning: The default base_image used by the @dsl.component decorator will switch from 'python:3.11' to 'python:3.12' on Oct 1, 2027. To ensure your existing components work with versions of the KFP SDK released after that date, you should provide an explicit base_image argument and ensure your component works as intended on Python 3.12.
  @component(


### Component 7 — `evaluate_model`

Runs inference on the validation split, logs F1/precision/recall/accuracy to Vertex Metrics, and saves a predictions CSV and evaluation report.

In [78]:
@component(
    packages_to_install=[
        "pandas",
        "numpy",
        "scikit-learn",
        "joblib"
    ]
)
def evaluate_model(
    input_dataset: Input[Dataset],
    model_artifact: Input[Artifact],
    metrics: Output[Metrics],
    predictions_dataset: Output[Dataset],
    evaluation_report: Output[Artifact],
    target_col: str = "target",
    id_col: str = "patient_id"
):
    import os
    import json
    import joblib
    import pandas as pd
    from sklearn.metrics import (
        f1_score,
        precision_score,
        recall_score,
        confusion_matrix,
        accuracy_score
    )

    # -----------------------------
    # Load validation dataset
    # -----------------------------
    df = pd.read_csv(input_dataset.path)

    print(f"Validation dataset shape: {df.shape}")

    # -----------------------------
    # Load trained model + metadata
    # -----------------------------
    model_path = os.path.join(model_artifact.path, "random_forest_model.joblib")
    metadata_path = os.path.join(model_artifact.path, "model_metadata.json")

    if not os.path.exists(model_path):
        raise ValueError(f"Missing model file: {model_path}")

    if not os.path.exists(metadata_path):
        raise ValueError(f"Missing model metadata file: {metadata_path}")

    model = joblib.load(model_path)

    with open(metadata_path, "r") as f:
        model_metadata = json.load(f)

    feature_cols = model_metadata["feature_cols"]

    # -----------------------------
    # Build validation feature matrix
    # -----------------------------
    X_val = df.reindex(columns=feature_cols, fill_value=0)
    y_val = df[target_col]

    print(f"Number of validation rows: {len(df)}")
    print(f"Number of model features expected: {len(feature_cols)}")

    # -----------------------------
    # Predict
    # -----------------------------
    y_pred = model.predict(X_val)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_val)[:, 1]
    else:
        y_prob = None

    # -----------------------------
    # Compute evaluation metrics
    # -----------------------------
    f1 = f1_score(y_val, y_pred, zero_division=0)
    precision = precision_score(y_val, y_pred, zero_division=0)
    recall = recall_score(y_val, y_pred, zero_division=0)
    accuracy = accuracy_score(y_val, y_pred)

    cm = confusion_matrix(y_val, y_pred)
    tn, fp, fn, tp = cm.ravel()

    print("Evaluation metrics:")
    print(f"F1 Score:   {f1:.4f}")
    print(f"Precision:  {precision:.4f}")
    print(f"Recall:     {recall:.4f}")
    print(f"Accuracy:   {accuracy:.4f}")
    print("Confusion Matrix:")
    print(cm)

    # -----------------------------
    # Log metrics for Vertex / KFP
    # -----------------------------
    metrics.log_metric("f1_score", float(f1))
    metrics.log_metric("precision", float(precision))
    metrics.log_metric("recall", float(recall))
    metrics.log_metric("accuracy", float(accuracy))
    metrics.log_metric("true_negatives", int(tn))
    metrics.log_metric("false_positives", int(fp))
    metrics.log_metric("false_negatives", int(fn))
    metrics.log_metric("true_positives", int(tp))

    # -----------------------------
    # Save predictions dataset
    # -----------------------------
    predictions_df = pd.DataFrame({
        id_col: df[id_col],
        "actual": y_val,
        "predicted": y_pred
    })

    if y_prob is not None:
        predictions_df["predicted_probability"] = y_prob

    predictions_df.to_csv(predictions_dataset.path, index=False)

    # -----------------------------
    # Save evaluation report artifact
    # -----------------------------
    os.makedirs(evaluation_report.path, exist_ok=True)

    report = {
        "model_type": model_metadata.get("model_type", "unknown"),
        "n_validation_rows": int(len(df)),
        "n_features_used": int(len(feature_cols)),
        "metrics": {
            "f1_score": float(f1),
            "precision": float(precision),
            "recall": float(recall),
            "accuracy": float(accuracy)
        },
        "confusion_matrix": {
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp)
        }
    }

    with open(os.path.join(evaluation_report.path, "evaluation_report.json"), "w") as f:
        json.dump(report, f, indent=2)

    print("Saved evaluation report contents:")
    print(os.listdir(evaluation_report.path))

/var/folders/9w/27dwnd7s3plbp9xj1v9qrn5m0000gn/T/ipykernel_43722/2424818708.py:1: FutureWarning: The default base_image used by the @dsl.component decorator will switch from 'python:3.11' to 'python:3.12' on Oct 1, 2027. To ensure your existing components work with versions of the KFP SDK released after that date, you should provide an explicit base_image argument and ensure your component works as intended on Python 3.12.
  @component(


---
## 7. Pipeline Assembly & Deployment

Assembles the components into a KFP pipeline, compiles to JSON, and submits to Vertex AI Pipelines.

In [79]:
from kfp.v2 import dsl
from kfp.dsl import pipeline


@pipeline(name="1.4.x-readmissions-smote-training-pipeline")
def readmissions_smote_training_pipeline(training_dataset_path: str):

    # 1. load data set
    validated_data_task = load_validate_data(input_dataset_path=training_dataset_path)

    # 2. split dataset
    split_task = split_data(input_dataset=validated_data_task.outputs["output_dataset"])

    # 3. oversample training
    oversampled_train_task = oversample_training(
        input_dataset=split_task.outputs["train_dataset"]
    )

    # 4. preprocess training
    preprocessed_train_task = fit_apply_preprocessing(
        input_dataset=oversampled_train_task.outputs["output_dataset"],
    )

    # 5. preprocess validation
    preprocessed_validation_task = apply_preprocessing(
        input_dataset=split_task.outputs["validation_dataset"],
        preprocessing_artifacts=preprocessed_train_task.outputs[
            "preprocessing_artifacts"
        ],
    )

    # 6. train and fit model
    trained_model_task = train_model(
        input_dataset=preprocessed_train_task.outputs["output_dataset"]
    )

    # 7. eval model
    evaluate_model(
        input_dataset=preprocessed_validation_task.outputs["output_dataset"],
        model_artifact=trained_model_task.outputs["model_artifact"],
    )

In [80]:
from kfp import compiler
TRAINING_PIPELINE_JSON = "readmissions_smote_training_pipeline.json"

compiler.Compiler().compile(
    pipeline_func=readmissions_smote_training_pipeline,
    package_path=TRAINING_PIPELINE_JSON
)

In [81]:
dataset_folder_uri = f"{BUCKET_ROOT_URI}/datasets/readmissions"
artifact_base_uri = f"{BUCKET_ROOT_URI}/training_pipeline_artifacts"
model_version = "v0"

In [82]:
from time import time


training_dataset_path = f"{dataset_folder_uri}/v1.4/train.csv"

training_job = aiplatform.PipelineJob(
    job_id=f"readmissions-smote-training-pipeline-{model_version}-{int(time())}",
    display_name=f"readmissions-1.4-smote-{model_version}",
    template_path=TRAINING_PIPELINE_JSON,
    pipeline_root=BUCKET_ROOT_URI,
    parameter_values={
        "training_dataset_path": training_dataset_path,
    },
    enable_caching=True,
    project=PROJECT_ID,
    location=LOCATION
)

training_job.submit(
)



Creating PipelineJob
PipelineJob created. Resource name: projects/182027088454/locations/us-central1/pipelineJobs/readmissions-smote-training-pipeline-v0-1777670428
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/182027088454/locations/us-central1/pipelineJobs/readmissions-smote-training-pipeline-v0-1777670428')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/readmissions-smote-training-pipeline-v0-1777670428?project=182027088454


In [83]:
log_pipeline_run(
    pipeline_job=training_job,
    dataset_version="v1.4.0",
    training_dataset_path=training_dataset_path,
    model_version=model_version,
    PROJECT_ID=PROJECT_ID,
    LOCATION=LOCATION,
    wait_for_completion=True,
)

Associating projects/182027088454/locations/us-central1/metadataStores/default/contexts/readmissions-pipeline-runs-pipeline-run-v0-20260501212029 to Experiment: readmissions-pipeline-runs


Waiting for pipeline to complete...
PipelineJob run completed. Resource name: projects/182027088454/locations/us-central1/pipelineJobs/readmissions-smote-training-pipeline-v0-1777670428
Logged eval metrics: {'precision': 0.5540229885057472, 'true_negatives': 1120.0, 'recall': 0.8731884057971014, 'accuracy': 0.8559748427672956, 'true_positives': 241.0, 'false_positives': 194.0, 'f1_score': 0.6779184247538678, 'false_negatives': 35.0}
Logged pipeline run to Vertex Experiments: readmissions-pipeline-runs / pipeline-run-v0-20260501212029


---
## 8. 1.4.1 Dataset Model Comparisons 

Experimenting on what model to use on the 1.4.1 Dataset
- 1.4.1 is the name of the dataset created from the pipeline above, which includes feature engineering and preprocessing steps.

In [ ]:
# run pipeline locally to get preprocessed data
import kfp.local

# Initialize local runner (SubprocessRunner installs packages_to_install into a venv)
kfp.local.init(runner=kfp.local.SubprocessRunner(use_venv=True))

# Call the pipeline function directly — no PipelineJob, no GCS needed
readmissions_smote_training_pipeline(
    training_dataset_path="../data/interim/1.4-edited-data.csv"
)

14:32:53.985 - INFO - Running pipeline: '1-4-x-readmissions-smote-training-pipeline'
--------------------------------------------------------------------------------
14:32:53.993 - INFO - Executing task 'load-validate-data'
14:32:53.994 - INFO - Streamed logs:

    [KFP Executor 2026-05-01 14:33:05,904 INFO]: Looking for component `load_validate_data` in --component_module_path `/var/folders/9w/27dwnd7s3plbp9xj1v9qrn5m0000gn/T/tmp.MPwaKnjMV3/ephemeral_component.py`
    [KFP Executor 2026-05-01 14:33:05,904 INFO]: Loading KFP component "load_validate_data" from /var/folders/9w/27dwnd7s3plbp9xj1v9qrn5m0000gn/T/tmp.MPwaKnjMV3/ephemeral_component.py (directory "/var/folders/9w/27dwnd7s3plbp9xj1v9qrn5m0000gn/T/tmp.MPwaKnjMV3" and module name "ephemeral_component")
    [KFP Executor 2026-05-01 14:33:05,905 INFO]: Got executor_input:
    {
        "inputs": {
            "parameterValues": {
                "input_dataset_path": "../data/interim/1.4-edited-data.csv"
            }
        },
 

In [ ]:
train_df = pd.read_csv(
    "./local_outputs/1-4-x-readmissions-smote-training-pipeline-2026-05-01-14-32-53-985651/fit-apply-preprocessing/output_dataset"
)
val_df = pd.read_csv(
    "./local_outputs/1-4-x-readmissions-smote-training-pipeline-2026-05-01-14-32-53-985651/apply-preprocessing/output_dataset"
)


,patient_id,height_m,smoker,bmi,adjusted_weight_kg,has_diabetes,has_hypertension,number_of_prior_visits,medications_prescribed,length_of_stay_score,...,diet_type_High-fat,diet_type_Other,diet_type_Vegetarian,type_of_treatment_Minor Surgery,type_of_treatment_Other Treatment,age_group_26-40,age_group_41-65,age_group_66-80,age_group_81+,target
0,1000589,1.927267,False,-0.725575,0.441455,False,True,-0.203389,1,1.691084,...,False,False,True,False,False,True,False,False,False,0
1,1004757,-0.012920,False,0.837486,0.680330,False,False,-0.203389,1,-0.907655,...,False,True,False,False,False,False,True,False,False,0
2,1005789,0.957173,False,0.346238,0.808108,False,False,-0.203389,1,-0.907655,...,False,True,False,False,False,False,True,False,False,0
3,1004711,0.957173,True,-0.502281,0.080688,True,False,-0.203389,0,1.691084,...,False,False,False,False,True,False,False,False,True,1
4,1006129,-0.012920,False,-0.011033,-0.082197,False,False,-0.203389,1,-0.907655,...,False,False,False,False,False,False,False,True,False,0
